# Parameter estimation

A probabilistic ODE solver turns an ODE with unknown parameters into a Gaussian
model of the data. Its marginal likelihood is the probability of the data under
the ODE, the solver's numerical uncertainty and the measurement noise. It is a
differentiable function of the parameters, so it can be maximized, sampled or
evaluated like any other likelihood. This notebook fits two parameters of the
Lotka-Volterra equations to noisy data, checks the fit against the data, and
compares the three evidence channels of `marginal_loglik`.

The script
[`examples/parameter_estimation.py`](https://github.com/paufisch/ode_filters/blob/main/examples/parameter_estimation.py)
fits the single parameter of a decay in the same way.

In [ ]:
import logging
import time

import jax
import jax.numpy as np
import matplotlib.pyplot as plt
import numpy as onp
import optax
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.integrate import solve_ivp
from tueplots import bundles, cycler
from tueplots.constants.color import palettes

from ode_filters import (
    IWP,
    InferenceProblem,
    Measurement,
    ODEFilter,
    ODEInformation,
    PositiveReal,
    affine_filter,
    conditional_loglik,
    fenrir_loglik,
    fit,
    fit_scipy,
    gaussian_filter,
    marginal_loglik,
    nis,
    observation_predictive,
    prepare_observations,
    residual_autocorrelation,
    rts_smoother,
    taylor_mode_initialization,
    taylor_surrogate,
    whitened_residuals,
)

jax.config.update("jax_enable_x64", True)

plt.rcParams.update(bundles.tue_ai_beamer())
plt.rcParams.update(cycler.cycler(color=palettes.paultol_bright))
set_matplotlib_formats("svg")  # vector figures stay sharp at any size
# Fall back to the default font quietly where Roboto Condensed is not installed.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
REFERENCE = palette[6]  # grey: true values and other references


## The data

Prey $x$ and predators $y$ follow

$$\dot x = \alpha x - \beta x y, \qquad \dot y = \delta x y - \gamma y.$$

The interaction rates $\beta = 0.5$ and $\delta = 0.25$ are known. The growth
rate of the prey $\alpha$ and the death rate of the predators $\gamma$ are not.
Both populations are counted every 0.2 time units, with noise of standard
deviation 0.15. The true trajectory comes from SciPy at the tolerances
`rtol = atol = 1e-11`.

In [ ]:
BETA, DELTA = 0.5, 0.25  # known
ALPHA, GAMMA = 1.0, 0.75  # to be estimated
X0 = np.array([2.0, 1.0])
NOISE = 0.15

ts = np.linspace(0.0, 20.0, 201)  # the solver's grid, step 0.1
t_obs = ts[2::2]  # every second node


def lotka_volterra(t, z, alpha, gamma):
    x, y = z
    return [alpha * x - BETA * x * y, DELTA * x * y - gamma * y]


truth = solve_ivp(
    lotka_volterra,
    (0.0, 20.0),
    onp.asarray(X0),
    args=(ALPHA, GAMMA),
    t_eval=onp.asarray(t_obs),
    rtol=1e-11,
    atol=1e-11,
).y.T
y = truth + NOISE * jax.random.normal(jax.random.PRNGKey(0), truth.shape)

fig, ax = plt.subplots()
for i, name in enumerate(["prey $x$", "predators $y$"]):
    ax.plot(t_obs, truth[:, i], color=REFERENCE, lw=2.5)
    ax.plot(t_obs, y[:, i], "o", ms=2.5, color=palette[i], label=name)
ax.plot([], [], color=REFERENCE, lw=2.5, label="true trajectory")
ax.set_xlabel("$t$")
ax.set_ylabel("population")
ax.legend(loc="upper right", ncols=3)
ax.set_ylim(top=1.25 * float(y.max()))  # room for the legend
plt.show()

## The model

An `InferenceProblem` maps the parameters $\theta = (\alpha, \gamma)$ to the
inputs of the solver: `build(theta)` returns the initial state and the ODE
information. The initial derivatives depend on $\theta$, so `build` computes
them by Taylor-mode differentiation. The grid, the prior and the calibration
stay fixed. The calibration is `"none"`, the recommended setting for inference,
so that the scale of the likelihood does not change with the parameters.

The data enter as an `ObsModel`. `prepare_observations` places each
measurement at its grid node, and every measurement time must be a node.

In [ ]:
prior = IWP(q=2, d=2)
data = prepare_observations(
    [Measurement(np.eye(2), y, t_obs, noise=NOISE**2)], prior.E0, ts
)


def build(theta):
    def vf(z, *, t):
        x, y = z
        return np.array(
            [theta["alpha"] * x - BETA * x * y, DELTA * x * y - theta["gamma"] * y]
        )

    m_0, P_0_sqr = taylor_mode_initialization(vf, X0, q=2)
    return m_0, P_0_sqr, ODEInformation(vf, prior.E0, prior.E1)


problem = InferenceProblem(build=build, prior=prior, ts=ts)


@jax.jit
def loglik(alpha, gamma):
    return marginal_loglik({"alpha": alpha, "gamma": gamma}, data, model=problem)


print(f"log-likelihood at the true parameters: {loglik(ALPHA, GAMMA):.2f}")

## Fitting

`fit_scipy` minimizes an objective of a parameter pytree with SciPy's
quasi-Newton method L-BFGS-B, using gradients from JAX. With the rates wrapped
in `PositiveReal`, the optimizer works on unconstrained values (through a
softplus) and cannot propose a negative rate. The initial guess,
$\alpha = 0.8$ and $\gamma = 0.9$, deviates by 20% from each true value:

In [ ]:
def objective(theta):
    return -marginal_loglik(theta, data, model=problem)


theta_hat, optimization = fit_scipy(
    objective, {"alpha": PositiveReal(0.8), "gamma": PositiveReal(0.9)}
)
alpha_hat = float(theta_hat["alpha"].unwrap())
gamma_hat = float(theta_hat["gamma"].unwrap())
print(
    f"alpha = {alpha_hat:.4f} (true {ALPHA}),  gamma = {gamma_hat:.4f} (true {GAMMA})"
)
print(f"{optimization.nfev} evaluations of the likelihood and its gradient")

For comparison, the cell below runs the same fit with the object API. An
`ODEFilter` holds the vector field with its trainable `ode_params`, and `fit`
runs an Optax optimizer for a fixed number of steps. On a smooth,
low-dimensional problem such as this one, a first-order method such as Adam
needs more evaluations than L-BFGS-B (here 300 steps against 11 evaluations),
but first-order methods scale to many parameters and to stochastic
objectives.

In [ ]:
def vf_params(z, params, *, t):
    x, y = z
    return np.array([params[0] * x - BETA * x * y, DELTA * x * y - params[1] * y])


def init_fn(params):
    return taylor_mode_initialization(
        lambda z, *, t: vf_params(z, params, t=t), X0, q=2
    )


model = ODEFilter(
    vf=vf_params, init_fn=init_fn, prior=prior, ts=ts, ode_params=np.array([0.8, 0.9])
)
fitted, losses = fit(model, data, optax.adam(0.01), steps=300)
print(
    f"Adam after {losses.shape[0]} steps: alpha, gamma = {onp.round(onp.asarray(fitted.ode_params), 4)}"
)

## The likelihood around the estimate

`marginal_loglik` is a pure JAX function, so `jax.vmap` evaluates it on a grid
of parameters in one call. Relative to its maximum, the contours at $-1.15$
and $-3.0$ bound the 68% and 95% likelihood-ratio regions for two parameters.
The two rates are negatively correlated. The period of the oscillation depends
approximately on their product, so a larger $\alpha$ partly compensates for a
smaller $\gamma$. The true values lie inside the 68% region:

In [ ]:
alphas = np.linspace(0.988, 1.016, 81)
gammas = np.linspace(0.738, 0.758, 81)
landscape = jax.vmap(lambda a: jax.vmap(lambda g: loglik(a, g))(gammas))(alphas)
relative = landscape - landscape.max()

fig, ax = plt.subplots()
ax.contour(
    gammas,
    alphas,
    relative,
    levels=[-3.0, -1.15],
    colors=palette[0],
    linestyles=["--", "-"],
)
ax.plot([], [], color=palette[0], label="68% region")
ax.plot([], [], color=palette[0], ls="--", label="95% region")
ax.plot(GAMMA, ALPHA, "+", color=REFERENCE, ms=10, mew=2, label="true value")
ax.plot(gamma_hat, alpha_hat, "x", color=palette[1], ms=6, mew=1.5, label="estimate")
ax.set_xlabel(r"$\gamma$")
ax.set_ylabel(r"$\alpha$")
ax.legend(loc="upper right")
plt.show()

## The fitted model against the data

At the estimate, a filter pass over the data and the smoother give the
posterior over the trajectory. `observation_predictive` maps it through the
measurement model. Without the noise, the predictive describes the uncertainty
about the trajectory. With the noise, it is the distribution of a new
measurement.

In [ ]:
def solve(alpha, gamma, noise=NOISE):
    """A filter pass with the data, assuming the given noise level."""
    obs = prepare_observations(
        [Measurement(np.eye(2), y, t_obs, noise=noise**2)], prior.E0, ts
    )
    m_0, P_0_sqr, measure = build({"alpha": alpha, "gamma": gamma})
    return gaussian_filter(
        m_0, P_0_sqr, prior, measure, ts, calibration="none", obs_model=obs
    )


result = solve(alpha_hat, gamma_hat)
posterior = rts_smoother(result)
trajectory = observation_predictive(posterior, data.H)
new_data = observation_predictive(posterior, data.H, data.R_sqr)

at_obs = onp.asarray(ts.searchsorted(t_obs))
inside = np.abs(y - new_data.m[at_obs]) <= 2.0 * new_data.std[at_obs]
print(f"std of the trajectory: {float(trajectory.std.mean()):.1e} (on average)")
print(f"std of a new measurement: {float(new_data.std.mean()):.3f}")
print(
    f"measurements inside the 2-std band: {100 * float(inside.mean()):.0f}% (expected about 95%)"
)

With the parameters fixed, the ODE determines the trajectory, and its standard
deviation is the solver's numerical uncertainty, about a hundred times smaller
than the noise. The spread of a new measurement therefore consists almost
entirely of the noise, and the band contains about the expected fraction of the
data:

In [ ]:
fig, ax = plt.subplots()
for i, name in enumerate(["prey $x$", "predators $y$"]):
    m, s = new_data.m[:, i], new_data.std[:, i]
    ax.fill_between(ts, m - 2 * s, m + 2 * s, color=palette[i], alpha=0.25, lw=0)
    ax.plot(ts, m, color=palette[i], label=name)
    ax.plot(t_obs, y[:, i], "o", ms=2.0, color=palette[i])
ax.set_xlabel("$t$")
ax.set_ylabel("population")
ax.legend(loc="upper right", ncols=2)
ax.set_ylim(top=1.2 * float(y.max()))
plt.show()

## Checking the fit

A high likelihood value does not show that the model is correct. The filter
provides a further test. Before each measurement it predicts the measurement,
as a Gaussian with mean `result.mz_obs` (the residual) and square-root
covariance `result.Pz_obs_sqr`. If the model is consistent with the data, the
residuals whitened by their predicted covariance, the **whitened residuals**,
are independent standard normal draws. We use two summaries of them:

- the **normalized innovation squared** (NIS), the squared norm of the whitened
  residual of a step, follows a $\chi^2$ distribution with as many degrees of
  freedom as the step has measurements, here with mean 2;
- the **autocorrelation** of the whitened residuals stays within about
  $\pm 2/\sqrt{n}$ of zero for $n$ measurements (a 95% range, so an
  occasional excursion is expected).

We compare the fit with two misspecified models: one that assumes a noise three
times smaller than the true one, and one with the growth rate of the prey 10%
too low.

In [ ]:
has_data = onp.asarray(data.mask.any(axis=1))  # the steps that carry a measurement
models = {
    "fit": solve(alpha_hat, gamma_hat),
    "noise too small": solve(alpha_hat, gamma_hat, noise=NOISE / 3),
    "alpha 10% low": solve(0.9 * alpha_hat, gamma_hat),
}
residuals = {}
for name, r in models.items():
    residuals[name] = whitened_residuals(r.mz_obs, r.Pz_obs_sqr)[has_data]
    rho = residual_autocorrelation(residuals[name], max_lag=1)[0]
    mean_nis = float(nis(r.mz_obs, r.Pz_obs_sqr)[has_data].mean())
    print(
        f"{name:16s} mean NIS {mean_nis:6.1f}   lag-1 autocorrelation (x, y): {onp.round(onp.asarray(rho), 2)}"
    )
print(f"{'':16s} expected: 2.0 and within +-{2 / onp.sqrt(has_data.sum()):.2f}")

The fit passes both checks. The noise level that is too small inflates the NIS
ninefold and leaves the residuals uncorrelated, since only the noise level of
the model is wrong. The wrong rate makes the residuals large and correlated in
time, because the model's trajectory drifts away from the data and stays away:

In [ ]:
n_obs = residuals["fit"].shape[0]
lags = onp.arange(1, 11)
fig, (left, right) = plt.subplots(1, 2)
for name, color in (("fit", palette[0]), ("alpha 10% low", palette[1])):
    left.plot(t_obs, residuals[name][:, 0], "o", ms=2.0, color=color, label=name)
    rho = residual_autocorrelation(residuals[name], max_lag=10)[:, 0]
    right.plot(lags, rho, "o-", ms=3, color=color)
for ax, bound in ((left, 2.0), (right, 2.0 / onp.sqrt(n_obs))):
    ax.axhspan(-bound, bound, color=REFERENCE, alpha=0.2, lw=0)
left.set_xlabel("$t$")
left.set_ylabel("whitened residual of $x$")
right.set_xlabel("lag")
right.set_ylabel("autocorrelation")
left.legend(loc="lower left")
plt.show()

The grey bands mark the approximate 95% ranges of a consistent model. The
residuals of the predators show the same pattern. Each check costs one filter
pass, and it detects misfits that the likelihood value alone does not rank.

## Comparing the evidence channels

`marginal_loglik` returns one of three evidences of the data, selected by
`channel=`. We write $Z$ for the ODE information at the grid nodes (observed
as zero) and $Y$ for the data. The channels differ in what each datum is
conditioned on:

- **`"obs"`** (the default, used above) is the sequential evidence
  $\sum_n \log p(y_n \mid z_{\le n}, y_{<n})$ of the filter. It conditions
  each datum on the ODE information up to the datum's time and on the earlier
  data. One evaluation runs one filter pass with the data.
- **`"fenrir"`** is the conditional evidence $\log p(Y \mid Z)$, which
  conditions every datum on all of the ODE information, including the part
  after the datum's time. Fenrir (Tronarp, Bosch & Hennig, 2022) runs one pass
  without the data, then a Kalman filter backwards through the backward
  conditionals of that pass, which updates on the data.
- **`"dalton"`** estimates the same $\log p(Y \mid Z)$ as the difference
  $\log p(Z, Y) - \log p(Z)$ of a pass with the data and a pass without them
  (Wu & Lysy, 2024).

We profile each channel over $\alpha$, with $\gamma$ at its estimate, for two
priors on the same grid: the `IWP(q=2)` of this notebook and `IWP(q=1)`, which
models one derivative fewer. `make_problem` rebuilds the problem above for a
given order. At the true parameters, the cell also compares two passes with
the true trajectory at the data times: the smoothed solve without data, and
the predictions of the filter pass with data.

In [ ]:
CHANNELS = ("obs", "fenrir", "dalton")
alphas_1d = np.linspace(0.985, 1.01, 101)


def make_problem(q):
    """The problem of this notebook with an IWP prior of order q, and its data."""
    prior_q = IWP(q=q, d=2)

    def build_q(theta):
        def vf(z, *, t):
            x, y = z
            return np.array(
                [theta["alpha"] * x - BETA * x * y, DELTA * x * y - theta["gamma"] * y]
            )

        m_0, P_0_sqr = taylor_mode_initialization(vf, X0, q=q)
        return m_0, P_0_sqr, ODEInformation(vf, prior_q.E0, prior_q.E1)

    data_q = prepare_observations(
        [Measurement(np.eye(2), y, t_obs, noise=NOISE**2)], prior_q.E0, ts
    )
    return InferenceProblem(build=build_q, prior=prior_q, ts=ts), data_q


def profile(problem_q, data_q, channel):
    """The log-likelihood of one channel over alphas_1d, gamma at its estimate."""

    def loglik_alpha(alpha):
        theta = {"alpha": alpha, "gamma": gamma_hat}
        return marginal_loglik(theta, data_q, model=problem_q, channel=channel)

    return jax.jit(jax.vmap(loglik_alpha))(alphas_1d)


problems = {q: make_problem(q) for q in (2, 1)}
profiles = {}
for q, (problem_q, data_q) in problems.items():
    for channel in CHANNELS:
        profiles[q, channel] = ll = profile(problem_q, data_q, channel)
        i = int(np.argmax(ll))
        print(
            f"q={q}  {channel:6s}  maximum {float(ll[i]):6.2f} at alpha = {float(alphas_1d[i]):.4f}"
        )

    # At the true parameters: errors against the true trajectory at the data times.
    m_0, P_0_sqr, measure = problem_q.build({"alpha": ALPHA, "gamma": GAMMA})
    args = (m_0, P_0_sqr, problem_q.prior, measure, ts)
    no_data = observation_predictive(
        rts_smoother(gaussian_filter(*args, calibration="none")), data_q.H
    )
    error = np.abs(no_data.m[at_obs] - truth)
    with_data = gaussian_filter(*args, calibration="none", obs_model=data_q)
    predicted = with_data.m_pred[at_obs - 1] @ data_q.H.T  # step i predicts node i + 1
    print(
        f"q={q}  largest error without data {float(error.max()):.3f} "
        f"({float((error / no_data.std[at_obs]).max()):.1f} std), "
        f"in the predictions with data {float(np.abs(predicted - truth).max()):.3f}\n"
    )

In [ ]:
fig, axes = plt.subplots(1, 2)
linestyles = {"obs": "-", "fenrir": "--", "dalton": ":"}
for ax, q in zip(axes, problems, strict=True):
    for color, channel in zip(palette, CHANNELS, strict=False):
        ll = profiles[q, channel]
        i = int(np.argmax(ll))
        ax.plot(alphas_1d, ll, color=color, ls=linestyles[channel], label=f'"{channel}"')
        ax.plot(alphas_1d[i], ll[i], "o", color=color, ms=4)
    ax.set_title(f"IWP($q = {q}$)")
    ax.set_xlabel(r"$\alpha$")
axes[0].set_ylabel("log-likelihood")
axes[0].legend(loc="lower right")
plt.show()

With `IWP(q=2)` the three profiles coincide, and all three maxima lie at the
estimate of the fit above, to the resolution of the grid. In this problem the
trajectory is a deterministic function of the parameters, so the three
channels would agree exactly for an exact solve. The solve without data
deviates from the true trajectory by at most 0.6 of its own standard
deviation, and the ODE information after a datum's time changes the evidence
by less than 0.2 over the range shown.

With `IWP(q=1)` the solve without data deviates from the true trajectory by up
to 1.6, six times its own standard deviation. The filter pass with data
conditions on each datum before it predicts the next one, and its predictions
at the data times stay within 0.2 of the true trajectory. The sequential
evidence judges each datum against these predictions. The conditional
evidence conditions every datum on all of the ODE information, which carries
the error of the whole solve. It lies 19 to 25 below the sequential evidence,
and the three maxima move apart. Since an exact solve makes the channels agree
in this problem, the gap between them points at a solver error larger than
the solver's uncertainty. Raising the order to `q=2` closes the gap (left
panel).

Fenrir and Dalton estimate the same quantity, and on one frozen linearization
of the ODE they agree exactly. The one-shot EK1 filter in `marginal_loglik`
linearizes the vector field at each step's prediction instead. Dalton's pass
with the data predicts other states than the pass without, so its two terms
belong to different linear models and its value differs from Fenrir's. With
`IWP(q=2)` both passes predict within 0.006 of the true trajectory, and the
two profiles differ by less than 0.002. With `IWP(q=1)` the pass without data
drifts away from the pass with data, the profiles differ by up to 5.2, and
their maxima lie 0.007 apart in $\alpha$.

The cell below freezes the EK1 linearization at the predictions of the pass
without data (`taylor_surrogate`) and runs both of Dalton's passes on the
frozen model (`affine_filter`). The frozen model reproduces the pass without
data, so Fenrir's value does not change, and Dalton's value now equals it:

In [ ]:
problem_1, data_1 = problems[1]
theta = {"alpha": alpha_hat, "gamma": gamma_hat}
m_0, P_0_sqr, measure = problem_1.build(theta)
no_data = gaussian_filter(m_0, P_0_sqr, problem_1.prior, measure, ts, calibration="none")

# The EK1 linearization at the predictions of the pass without data, frozen.
surrogate = taylor_surrogate(measure, no_data.m_pred, ts=ts[1:])
frozen_no_data = affine_filter(m_0, P_0_sqr, problem_1.prior, surrogate, ts)
frozen_data = affine_filter(
    m_0, P_0_sqr, problem_1.prior, surrogate, ts, obs_model=data_1
)

for channel in ("fenrir", "dalton"):
    one_shot = marginal_loglik(theta, data_1, model=problem_1, channel=channel)
    print(f"one-shot {channel}: {float(one_shot):.6f}")
print(f"frozen   fenrir: {float(fenrir_loglik(frozen_no_data, data_1)):.6f}")
print(f"frozen   dalton: {float(conditional_loglik(frozen_data, frozen_no_data)):.6f}")

The cost of a channel follows from its passes over the grid. The sequential
evidence runs one filter pass. Fenrir runs the filter pass without the data
and then a backward pass over the data, which does not evaluate the vector
field. Dalton runs two filter passes. The cell below times one jitted
evaluation of each channel for the problem of this notebook (200 steps,
`IWP(q=2)`); the times depend on the machine.

In [ ]:
def milliseconds_per_evaluation(channel, repeats=100):
    """Wall time of one jitted evaluation of a channel, at the estimate."""
    evaluate = jax.jit(
        lambda theta: marginal_loglik(theta, data, model=problem, channel=channel)
    )
    theta = {"alpha": alpha_hat, "gamma": gamma_hat}
    evaluate(theta).block_until_ready()  # compile once
    start = time.perf_counter()
    for _ in range(repeats):
        evaluate(theta).block_until_ready()
    return 1e3 * (time.perf_counter() - start) / repeats


for channel in CHANNELS:
    print(f"{channel:6s}  {milliseconds_per_evaluation(channel):.2f} ms")

In this notebook the default `"obs"` channel and the conditional evidence give
the same estimate, and `"obs"` costs one filter pass instead of two passes.
To compare hyperparameters, such as a prior's length scale or `Xi`, use the
conditional evidence, which judges the data against all of the ODE
information (see
[Selecting hyperparameters from data](../../calibration/#selecting-hyperparameters-from-data)).

## See also

- [Bayesian posterior sampling](../bayesian-sampling/): the full posterior over
  the parameters instead of a point estimate.
- [Recovering a hidden input](../hidden-input/): estimating an unknown forcing
  function instead of a finite number of parameters.
- [Selecting hyperparameters from data](../../calibration/#selecting-hyperparameters-from-data):
  which evidence to maximize, and the conditional evidence.
- The [inference API](../../api/inference/).